# Clickstream inspection and simulated batches

## Setup
Run from `data/notebooks/` using Python (clickstream). Input: `../raw/e-shop clothing 2008.csv`. Output: `../batches/`. Write cells replace same-named batch files. Daily arrivals are simulated.

Source: https://archive.ics.uci.edu/dataset/553/clickstream+data+for+online+shopping

Saved outputs were retained from earlier runs. Restart the kernel and Run All to validate this standalone notebook.

## Steps

### Locate the source file

In [1]:
# Use a path relative to this notebook folder.
from pathlib import Path
import pandas as pd

file_path = Path("../raw/e-shop clothing 2008.csv")


print("file exist：", file_path.exists())

file exist： True


### Inspect the CSV header

In [2]:
# Read the header to check the delimiter.
with file_path.open(encoding="utf-8-sig") as file:
    print(file.readline())

year;month;day;order;country;session ID;page 1 (main category);page 2 (clothing model);colour;location;model photography;price;price 2;page



### Load click records

In [3]:
# Use a semicolon to separate the columns.
df = pd.read_csv(file_path, sep=";")

df.head()

,year,month,day,order,country,session ID,page 1 (main category),page 2 (clothing model),colour,location,model photography,price,price 2,page
0,2008,4,1,1,29,1,1,A13,1,5,1,28,2,1
1,2008,4,1,2,29,1,1,A16,1,6,1,33,2,1
2,2008,4,1,3,29,1,2,B4,10,2,1,52,1,1
3,2008,4,1,4,29,1,2,B17,6,6,2,38,2,1
4,2008,4,1,5,29,1,2,B8,4,3,2,52,1,1


### Inspect the table structure

In [4]:
# Check row counts, column names, types, and non-null counts.
print("# of rows, # of columns: ", df.shape)
print("Name of columns: ", df.columns.tolist())

df.info()

# of rows, # of columns:  (165474, 14)
Name of columns:  ['year', 'month', 'day', 'order', 'country', 'session ID', 'page 1 (main category)', 'page 2 (clothing model)', 'colour', 'location', 'model photography', 'price', 'price 2', 'page']
<class 'pandas.DataFrame'>
RangeIndex: 165474 entries, 0 to 165473
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype
---  ------                   --------------   -----
 0   year                     165474 non-null  int64
 1   month                    165474 non-null  int64
 2   day                      165474 non-null  int64
 3   order                    165474 non-null  int64
 4   country                  165474 non-null  int64
 5   session ID               165474 non-null  int64
 6   page 1 (main category)   165474 non-null  int64
 7   page 2 (clothing model)  165474 non-null  str  
 8   colour                   165474 non-null  int64
 9   location                 165474 non-null  int64
 10  model photography  

### Check dates

In [5]:
# Combine year, month, and day without changing df. Invalid dates become NaT.
event_date = pd.to_datetime(
    df[["year", "month", "day"]],
    errors="coerce"
)

print("Invalid date count: ", event_date.isna().sum())
print("Earliest date: ", event_date.min())
print("Latest date:", event_date.max())
print("# of distinct dates:", event_date.nunique())

Invalid date count:  0
Earliest date:  2008-04-01 00:00:00
Latest date: 2008-08-13 00:00:00
# of distinct dates: 135


### Check duplicates

In [6]:
# Check complete rows and the proposed event key separately.
print("Exact duplicate rows:", df.duplicated().sum())

event_key = ["session ID", "order"]

print(
    "Duplicate event keys:",
    df.duplicated(subset=event_key).sum()
)

Exact duplicate rows: 0
Duplicate event keys: 0


### Check session dates

In [7]:
# Count distinct dates per session to find sessions spanning multiple days.
dates_per_session = (
    df.assign(event_date=event_date)
    .groupby("session ID")["event_date"]
    .nunique()
)

print("Total sessions:", dates_per_session.size)
print("Sessions spanning multiple dates:", (dates_per_session > 1).sum())
print("Maximum dates per session:", dates_per_session.max())

Total sessions: 24026
Sessions spanning multiple dates: 0
Maximum dates per session: 1


### Check click sequences

In [8]:
# Compare the sequence range with the row count; event keys are already unique.
session_order = (
    df.groupby("session ID")["order"]
    .agg(["min", "max", "count"])
)

print(
    "Sessions not starting at 1:",
    (session_order["min"] != 1).sum()
)

print(
    "Sessions with gaps in click order:",
    (
        session_order["max"] - session_order["min"] + 1
        != session_order["count"]
    ).sum()
)

session_order.head()

Sessions not starting at 1: 0
Sessions with gaps in click order: 0


,min,max,count
session ID,,,
1,1,9,9
2,1,10,10
3,1,6,6
4,1,4,4
5,1,1,1


### Check main categories

In [9]:
# Flag category codes outside the documented set.
category = df["page 1 (main category)"]

print("Category counts:")
print(category.value_counts().sort_index())

invalid_category = ~category.isin([1, 2, 3, 4])

print("Invalid category rows:", invalid_category.sum())

Category counts:
page 1 (main category)
1    49742
2    38408
3    38577
4    38747
Name: count, dtype: int64
Invalid category rows: 0


### Check other category codes

In [10]:
# Check each allowed set; range excludes its stop value.
allowed_values = {
    "country": range(1, 48),
    "colour": range(1, 15),
    "location": range(1, 7),
    "model photography": [1, 2],
    "price 2": [1, 2],
    "page": range(1, 6),
}

for column, allowed in allowed_values.items():
    invalid = ~df[column].isin(allowed)
    print(f"{column} - invalid rows: {invalid.sum()}")

country - invalid rows: 0
colour - invalid rows: 0
location - invalid rows: 0
model photography - invalid rows: 0
price 2 - invalid rows: 0
page - invalid rows: 0


### Check prices and products

In [11]:
# Prices are observed per click, so repeated views affect the mean.
print("Price summary:")
print(df["price"].describe())

print("Rows with zero price:", (df["price"] == 0).sum())
print("Rows with negative price:", (df["price"] < 0).sum())

product = df["page 2 (clothing model)"]

print("Distinct product codes:", product.nunique())
print("Missing product codes:", product.isna().sum())
print(
    "Blank product codes:",
    product.str.strip().eq("").sum()
)

Price summary:
count    165474.000000
mean         43.802507
std          12.548131
min          18.000000
25%          33.000000
50%          43.000000
75%          52.000000
max          82.000000
Name: price, dtype: float64
Rows with zero price: 0
Rows with negative price: 0
Distinct product codes: 217
Missing product codes: 0
Blank product codes: 0


## Initial data inspection findings

- The dataset contains 165,474 click records and 14 columns.
- Dates cover 2008-04-01 to 2008-08-13, with records on all 135 days.
- There are 24,026 distinct session IDs.
- No missing values were detected by pandas.
- No exact duplicate rows or duplicate (session ID, order) keys were found.
- Each session ID appears on only one date.
- Click sequences start at 1 and have no internal gaps.
- All checked categorical codes fall within the documented allowed values.
- Prices range from USD 18 to USD 82, with no zero or negative values.
- There are 217 distinct product codes, with no missing or blank codes.

## Initial design decisions

- Grain: one recorded product-browsing click per row.
- Candidate event key: (session ID, order).
- Batch unit: one event date per simulated batch.
- Daily arrival is simulated; actual arrival times are unavailable.
- These checks establish structural consistency, not complete source-log coverage.

### Build session summaries

In [12]:
# Create one row per session. Each session has only one date in this dataset.
session_summary = (
    df.assign(event_date=event_date)
    .groupby("session ID", as_index=False)
    .agg(
        event_date=("event_date", "first"),
        click_count=("order", "size"),
        distinct_product_count=("page 2 (clothing model)", "nunique"),
        distinct_category_count=("page 1 (main category)", "nunique"),
    )
    .rename(columns={"session ID": "session_id"})
)

session_summary.head()

,session_id,event_date,click_count,distinct_product_count,distinct_category_count
0,1,2008-04-01,9,9,4
1,2,2008-04-01,10,8,3
2,3,2008-04-01,6,6,3
3,4,2008-04-01,4,4,2
4,5,2008-04-01,1,1,1


### Check session totals

In [13]:
# Compare the summary with the source and check key uniqueness.
print("Summary rows:", len(session_summary))
print("Source sessions:", df["session ID"].nunique())

print("Total summarized clicks:", session_summary["click_count"].sum())
print("Source rows:", len(df))

print(
    "Duplicate session IDs:",
    session_summary["session_id"].duplicated().sum()
)

Summary rows: 24026
Source sessions: 24026
Total summarized clicks: 165474
Source rows: 165474
Duplicate session IDs: 0


### Build daily summaries

In [14]:
# Count products directly from clicks to avoid double counting across sessions.
daily_summary = (
    df.assign(event_date=event_date)
    .groupby("event_date", as_index=False)
    .agg(
        click_count=("order", "size"),
        session_count=("session ID", "nunique"),
        distinct_product_count=("page 2 (clothing model)", "nunique"),
    )
)

daily_summary.head()

,event_date,click_count,session_count,distinct_product_count
0,2008-04-01,3181,477,209
1,2008-04-02,3456,480,213
2,2008-04-03,1854,276,209
3,2008-04-04,1801,266,206
4,2008-04-05,1504,199,199


### Check daily totals

In [15]:
# Session counts can be added because sessions do not span dates here.
print("Summary days:", len(daily_summary))
print("Source dates:", event_date.nunique())

print("Total daily clicks:", daily_summary["click_count"].sum())
print("Source rows:", len(df))

print("Total daily sessions:", daily_summary["session_count"].sum())
print("Source sessions:", df["session ID"].nunique())

Summary days: 135
Source dates: 135
Total daily clicks: 165474
Source rows: 165474
Total daily sessions: 24026
Source sessions: 24026


### Select the first batch

In [16]:
# Select all records on the earliest date as a simulated daily delivery.
events = df.assign(event_date=event_date)

batch_date = events["event_date"].min()

first_batch = events.loc[
    events["event_date"] == batch_date
].copy()

print("Batch date:", batch_date.date())
print("Batch rows:", len(first_batch))
print("Batch sessions:", first_batch["session ID"].nunique())

first_batch.head()

Batch date: 2008-04-01
Batch rows: 3181
Batch sessions: 477


,year,month,day,order,country,session ID,page 1 (main category),page 2 (clothing model),colour,location,model photography,price,price 2,page,event_date
0,2008,4,1,1,29,1,1,A13,1,5,1,28,2,1,2008-04-01
1,2008,4,1,2,29,1,1,A16,1,6,1,33,2,1,2008-04-01
2,2008,4,1,3,29,1,2,B4,10,2,1,52,1,1,2008-04-01
3,2008,4,1,4,29,1,2,B17,6,6,2,38,2,1,2008-04-01
4,2008,4,1,5,29,1,2,B8,4,3,2,52,1,1,2008-04-01


### Check the first batch

In [17]:
# Compare this batch with the daily summary for the same date.
expected = daily_summary.loc[
    daily_summary["event_date"] == batch_date
].iloc[0]

print(
    "Click count matches:",
    len(first_batch) == expected["click_count"]
)
print(
    "Session count matches:",
    first_batch["session ID"].nunique() == expected["session_count"]
)

Click count matches: True
Session count matches: True


### Save the first batch

In [18]:
# Create the output folder. Rerunning replaces the same generated file.
batch_dir = Path("../batches")
batch_dir.mkdir(parents=True, exist_ok=True)

batch_path = batch_dir / f"clickstream_{batch_date:%Y-%m-%d}.csv"

# Keep the original 14 columns in the simulated input
batch_output = first_batch[df.columns]

batch_output.to_csv(
    batch_path,
    sep=";",
    index=False,
    encoding="utf-8"
)

print("Saved batch:", batch_path)
print("Saved rows:", len(batch_output))

Saved batch: ..\batches\clickstream_2008-04-01.csv
Saved rows: 3181


### Read back the first batch

In [19]:
# Compare values and types after saving; CSV does not store the row index.
batch_loaded = pd.read_csv(batch_path, sep=";")

print("Loaded rows:", len(batch_loaded))
print("Loaded columns:", len(batch_loaded.columns))

pd.testing.assert_frame_equal(
    batch_output.reset_index(drop=True),
    batch_loaded
)

print("Batch round-trip check passed.")

Loaded rows: 3181
Loaded columns: 14
Batch round-trip check passed.


### Save all daily batches

In [20]:
# Write one file per date with the original 14 columns.
for current_date, batch in events.groupby("event_date"):
    batch_path = batch_dir / f"clickstream_{current_date:%Y-%m-%d}.csv"

    batch[df.columns].to_csv(
        batch_path,
        sep=";",
        index=False,
        encoding="utf-8"
    )

print("Daily batches written:", events["event_date"].nunique())

Daily batches written: 135


### Verify all daily batches

In [21]:
# Read every generated file and compare it with the source records.
total_rows = 0
checked_batches = 0

for current_date, batch in events.groupby("event_date"):
    batch_path = batch_dir / f"clickstream_{current_date:%Y-%m-%d}.csv"
    loaded = pd.read_csv(batch_path, sep=";")

    pd.testing.assert_frame_equal(
        batch[df.columns].reset_index(drop=True),
        loaded
    )

    total_rows += len(loaded)
    checked_batches += 1

print("Verified batches:", checked_batches)
print("Total batch rows:", total_rows)
print("Source rows:", len(df))
print("All batch round-trip checks passed.")

Verified batches: 135
Total batch rows: 165474
Source rows: 165474
All batch round-trip checks passed.


## Next steps
Open `02_spark_single_batch.ipynb` to read the first batch with Spark.